In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/former_names.csv
/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/goalscorers.csv
/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/shootouts.csv
/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/results.csv


In [2]:
!pip install lightgbm statsmodels -q

In [4]:
import pandas as pd
import numpy as np

INPUT_CSV = "/kaggle/input/datasets/martj42/international-football-results-from-1872-to-2017/results.csv"
OUTPUT_CSV = "/kaggle/working/results_with_elo.csv"
INITIAL_ELO = 1500
BASE_K = 30
WORLD_CUP_K = 45
HOME_ADVANTAGE = 60

def load_data(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df["home_team"] = df["home_team"].str.strip()
    df["away_team"] = df["away_team"].str.strip()
    return df

def get_k_factor(tournament):
    if isinstance(tournament, str) and "World Cup" in tournament:
        return WORLD_CUP_K
    return BASE_K

def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def compute_elo(df):
    elo_dict = {}
    home_elo_before, away_elo_before = [], []
    for idx, row in df.iterrows():
        home, away = row["home_team"], row["away_team"]
        elo_dict.setdefault(home, INITIAL_ELO)
        elo_dict.setdefault(away, INITIAL_ELO)
        elo_h, elo_a = elo_dict[home], elo_dict[away]
        home_elo_before.append(elo_h)
        away_elo_before.append(elo_a)
        if row["home_score"] > row["away_score"]:
            score_h, score_a = 1, 0
        elif row["home_score"] < row["away_score"]:
            score_h, score_a = 0, 1
        else:
            score_h, score_a = 0.5, 0.5
        adj_home = 0 if row.get("neutral", False) else HOME_ADVANTAGE
        exp_h = expected_score(elo_h + adj_home, elo_a)
        exp_a = 1 - exp_h
        k = get_k_factor(row.get("tournament", ""))
        elo_dict[home] = elo_h + k * (score_h - exp_h)
        elo_dict[away] = elo_a + k * (score_a - exp_a)
    df["home_elo"] = home_elo_before
    df["away_elo"] = away_elo_before
    df["elo_diff"] = df["home_elo"] - df["away_elo"]
    return df, elo_dict

def add_recent_form(df, n_matches=5):
    team_history = {}
    home_avg_gf, home_avg_ga, home_form = [], [], []
    away_avg_gf, away_avg_ga, away_form = [], [], []
    for idx, row in df.iterrows():
        for side, team, gf_col, ga_col, avg_gf_list, avg_ga_list, form_list in [
            ("home", row["home_team"], "home_score", "away_score", home_avg_gf, home_avg_ga, home_form),
            ("away", row["away_team"], "away_score", "home_score", away_avg_gf, away_avg_ga, away_form),
        ]:
            hist = team_history.get(team, [])
            recent = hist[-n_matches:] if len(hist) > 0 else []
            if recent:
                avg_gf_list.append(np.mean([m[0] for m in recent]))
                avg_ga_list.append(np.mean([m[1] for m in recent]))
                form_list.append(np.mean([m[2] for m in recent]))
            else:
                avg_gf_list.append(np.nan)
                avg_ga_list.append(np.nan)
                form_list.append(np.nan)
        h_result = 1 if row["home_score"] > row["away_score"] else (0.5 if row["home_score"] == row["away_score"] else 0)
        team_history.setdefault(row["home_team"], []).append((row["home_score"], row["away_score"], h_result))
        team_history.setdefault(row["away_team"], []).append((row["away_score"], row["home_score"], 1 - h_result if h_result != 0.5 else 0.5))
    df["home_avg_gf"] = home_avg_gf
    df["home_avg_ga"] = home_avg_ga
    df["home_form"] = home_form
    df["away_avg_gf"] = away_avg_gf
    df["away_avg_ga"] = away_avg_ga
    df["away_form"] = away_form
    return df

def add_label(df):
    conditions = [df["home_score"] > df["away_score"], df["home_score"] == df["away_score"], df["home_score"] < df["away_score"]]
    choices = ["home_win", "draw", "away_win"]
    df["result"] = np.select(conditions, choices, default="unknown")
    return df

df = load_data(INPUT_CSV)
df, final_elo = compute_elo(df)
df = add_recent_form(df, n_matches=5)
df = add_label(df)
df = df.dropna(subset=["home_avg_gf", "away_avg_gf"])
df.to_csv(OUTPUT_CSV, index=False)
print(f"完成！輸出至 {OUTPUT_CSV}，共 {len(df)} 筆比賽資料")

elo_ranking = pd.Series(final_elo).sort_values(ascending=False)
elo_ranking.to_csv("/kaggle/working/latest_elo_ranking.csv", header=["elo"])
print("最新 Elo 排行榜已存至 /kaggle/working/latest_elo_ranking.csv")

完成！輸出至 /kaggle/working/results_with_elo.csv，共 49203 筆比賽資料
最新 Elo 排行榜已存至 /kaggle/working/latest_elo_ranking.csv


In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, classification_report
import statsmodels.api as sm
import statsmodels.formula.api as smf
import joblib

INPUT_CSV = "/kaggle/working/results_with_elo.csv"

FEATURE_COLS = [
    "elo_diff", "home_elo", "away_elo",
    "home_avg_gf", "home_avg_ga", "home_form",
    "away_avg_gf", "away_avg_ga", "away_form",
]

df = pd.read_csv(INPUT_CSV, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df[df["result"] != "unknown"].reset_index(drop=True)

split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print(f"訓練集: {len(train_df)} 筆 | 測試集: {len(test_df)} 筆")

le = LabelEncoder()
y_train = le.fit_transform(train_df["result"])
y_test = le.transform(test_df["result"])

X_train = train_df[FEATURE_COLS]
X_test = test_df[FEATURE_COLS]

clf = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
)

clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

pred = clf.predict(X_test)
pred_proba = clf.predict_proba(X_test)

print("\n=== LightGBM 分類結果 ===")
print("Accuracy:", accuracy_score(y_test, pred))
print("Log Loss:", log_loss(y_test, pred_proba))
print(classification_report(y_test, pred, target_names=le.classes_))

joblib.dump(clf, "/kaggle/working/model_classifier.pkl")
joblib.dump(le, "/kaggle/working/label_encoder.pkl")

poisson_rows = []
for _, row in train_df.iterrows():
    poisson_rows.append({
        "goals": row["home_score"], "team_elo": row["home_elo"], "opp_elo": row["away_elo"],
        "team_avg_gf": row["home_avg_gf"], "opp_avg_ga": row["away_avg_ga"], "is_home": 1,
    })
    poisson_rows.append({
        "goals": row["away_score"], "team_elo": row["away_elo"], "opp_elo": row["home_elo"],
        "team_avg_gf": row["away_avg_gf"], "opp_avg_ga": row["home_avg_ga"], "is_home": 0,
    })

poisson_df = pd.DataFrame(poisson_rows)

poisson_model = smf.glm(
    formula="goals ~ team_elo + opp_elo + team_avg_gf + opp_avg_ga + is_home",
    data=poisson_df,
    family=sm.families.Poisson(),
).fit()

print("\n=== 泊松迴歸模型摘要 ===")
print(poisson_model.summary())

joblib.dump(poisson_model, "/kaggle/working/model_poisson.pkl")
print("\n模型已存檔完成")

訓練集: 44279 筆 | 測試集: 4920 筆
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003274 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1098
[LightGBM] [Info] Number of data points in the train set: 44279, number of used features: 9
[LightGBM] [Info] Start training from score -1.268951
[LightGBM] [Info] Start training from score -1.478074
[LightGBM] [Info] Start training from score -0.711725
Training until validation scores don't improve for 30 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

In [7]:
import pandas as pd
import numpy as np
import joblib
from scipy.stats import poisson

N_SIMULATIONS = 20000
MAX_GOALS = 8

poisson_model = joblib.load("/kaggle/working/model_poisson.pkl")
elo_ranking = pd.read_csv("/kaggle/working/latest_elo_ranking.csv", index_col=0)["elo"].to_dict()

QUARTERFINALS = [
    ("Morocco", "France"),
    ("England", "Norway"),
    ("Spain", "Belgium"),
    ("Argentina", "Switzerland"),
]

def get_elo(team):
    for key in elo_ranking:
        if key.lower() == team.lower():
            return elo_ranking[key]
    print(f"⚠️ 找不到 {team} 的 Elo，使用預設值 1600")
    return 1600

def expected_goals(team_elo, opp_elo):
    X = pd.DataFrame([{
        "team_elo": team_elo, "opp_elo": opp_elo,
        "team_avg_gf": 1.5, "opp_avg_ga": 1.2, "is_home": 0,
    }])
    return poisson_model.predict(X)[0]

def match_win_prob(team_a, team_b):
    lam_a = expected_goals(get_elo(team_a), get_elo(team_b))
    lam_b = expected_goals(get_elo(team_b), get_elo(team_a))
    matrix = np.outer(
        poisson.pmf(np.arange(MAX_GOALS + 1), lam_a),
        poisson.pmf(np.arange(MAX_GOALS + 1), lam_b),
    )
    p_a = np.tril(matrix, -1).sum()
    p_draw = np.trace(matrix)
    p_b = np.triu(matrix, 1).sum()
    p_a_adj = p_a + p_draw * (p_a / (p_a + p_b))
    return p_a_adj

def simulate_knockout(team_a, team_b):
    p_a = match_win_prob(team_a, team_b)
    return np.random.choice([team_a, team_b], p=[p_a, 1 - p_a])

print("=== 8強單場勝率預估 ===")
for a, b in QUARTERFINALS:
    p = match_win_prob(a, b)
    print(f"{a} 勝率 {p:.1%}  vs  {b} 勝率 {1-p:.1%}")

champion_count = {}
final_count = {}
semifinal_count = {}

for _ in range(N_SIMULATIONS):
    semifinalists = [simulate_knockout(a, b) for a, b in QUARTERFINALS]
    for t in semifinalists:
        semifinal_count[t] = semifinal_count.get(t, 0) + 1
    finalists = [
        simulate_knockout(semifinalists[0], semifinalists[1]),
        simulate_knockout(semifinalists[2], semifinalists[3]),
    ]
    for t in finalists:
        final_count[t] = final_count.get(t, 0) + 1
    champion = simulate_knockout(finalists[0], finalists[1])
    champion_count[champion] = champion_count.get(champion, 0) + 1

results = pd.DataFrame({
    "team": list(champion_count.keys()),
    "champion_prob": [v / N_SIMULATIONS for v in champion_count.values()],
    "final_prob": [final_count.get(t, 0) / N_SIMULATIONS for t in champion_count.keys()],
    "semifinal_prob": [semifinal_count.get(t, 0) / N_SIMULATIONS for t in champion_count.keys()],
}).sort_values("champion_prob", ascending=False)

print("\n=== 2026世界盃奪冠機率模擬（基於8強現況）===")
print(results.to_string(index=False))

results.to_csv("/kaggle/working/champion_probabilities_qf.csv", index=False)

=== 8強單場勝率預估 ===
Morocco 勝率 38.4%  vs  France 勝率 61.6%
England 勝率 58.3%  vs  Norway 勝率 41.7%
Spain 勝率 66.5%  vs  Belgium 勝率 33.5%
Argentina 勝率 70.3%  vs  Switzerland 勝率 29.7%

=== 2026世界盃奪冠機率模擬（基於8強現況）===
       team  champion_prob  final_prob  semifinal_prob
  Argentina        0.24645     0.40170         0.70870
      Spain        0.21745     0.36330         0.66650
     France        0.18710     0.37340         0.61565
    England        0.12020     0.27705         0.58345
    Morocco        0.07440     0.18530         0.38435
     Norway        0.05760     0.16425         0.41655
    Belgium        0.05350     0.12775         0.33350
Switzerland        0.04330     0.10725         0.29130
